[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/maryrobynbernabe/coral-sound-project-3-model-training)

In [1]:
# Set the training dataset path
TRAIN_CSV = "/kaggle/input/notebooks/maryrobynbernabe/coral-sound-project-2-dataset-splitting/train_set.csv"

Visualize the data

In [2]:
import pandas as pd

# load the train csv
df_train = pd.read_csv(TRAIN_CSV)
df_train.head()

,audio_file_name,RMSE,SPECTRAL_CENTROID,SPECTRAL_BANDWIDTH,ROLLOFF,ZERO_CROSSING_RATE,MFCC_FEATURE_0,MFCC_FEATURE_1,MFCC_FEATURE_2,MFCC_FEATURE_3,...,MFCC_FEATURE_11,MFCC_FEATURE_12,MFCC_FEATURE_13,MFCC_FEATURE_14,MFCC_FEATURE_15,MFCC_FEATURE_16,MFCC_FEATURE_17,MFCC_FEATURE_18,MFCC_FEATURE_19,labels
0,BoN11.0529H.1678278701.180906.4.28.wav,0.013586,3281.713706,3527.383649,7962.826850,0.001039,-495.97717,-25.360006,15.571173,10.742051,...,8.197722,2.783091,7.597062,1.207888,7.518812,4.713681,7.761002,2.533650,7.588442,Healthy
1,BaF5.1330H.671907872.180831.3.18.wav,0.000624,5506.005847,3041.157454,8939.850109,0.254344,-611.53705,-16.766792,34.535435,31.588339,...,2.514375,-3.736288,2.271210,-1.622616,3.087977,-2.768877,3.200042,-0.567399,1.135935,Healthy
2,SaF5.1125R.1678278701.180831.5.33.wav,0.013555,2238.091512,3304.071066,6828.200377,0.000297,-581.68410,-27.747358,29.402313,28.410930,...,6.210212,3.216774,8.026078,1.109000,7.852259,3.715654,7.919775,3.877651,7.079189,Restored
3,SaF2.1112R.671907872.180828.5.49.wav,0.000631,5802.834647,2953.160951,9101.115800,0.419278,-584.80005,-26.345171,26.770414,24.583618,...,6.878403,0.387064,2.824635,-1.029802,4.852212,-1.872255,3.994614,0.126457,2.576796,Restored
4,SaF5.1202D.1677983769.180831.5.30.wav,0.010829,2637.516704,3333.849107,7146.860951,0.000484,-559.42035,-37.722710,3.701616,32.711998,...,6.353807,3.654529,7.881154,1.278509,7.627464,5.473676,8.769265,4.218484,9.438093,Degraded


Convert label column to numbers

In [3]:
# Define label column
LABEL_COLUMN = "labels"

print(df_train[LABEL_COLUMN].value_counts())

labels
Restored    88
Healthy     64
Degraded    57
Name: count, dtype: int64


In [4]:
# convert into numbers
df_train[LABEL_COLUMN] = pd.factorize(df_train[LABEL_COLUMN], sort = True)[0]

Separate features and labels

In [5]:
# gets all features without the labels and discrete features
cols_to_drop = ['audio_file_name', LABEL_COLUMN]
X = df_train.drop(columns=cols_to_drop, axis = 1).values

# all rows
# only the last column, which is the label
Y = df_train[LABEL_COLUMN].values

Training the model using cross validation

In [6]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Function to perform k-fold cross validation
def cross_validation(model, data = (X, Y), splits = 5):
    kf = KFold(n_splits=splits, shuffle=True, random_state=42)

    # Perform k-fold cross-validation
    accuracy = []
    precision = []
    recall = []
    f1 = []

    for train_index, valid_index in kf.split(data[0]):
        X_train, X_valid = data[0][train_index], data[0][valid_index]
        y_train, y_valid = data[1][train_index], data[1][valid_index]

        # Fit the defined model
        model.fit(X_train, y_train)

        # Make predictions on the test data
        y_pred = model.predict(X_valid)

        # Calculate accuracy, precision and recall
        accuracy.append(accuracy_score(y_valid, y_pred))
        precision.append(precision_score(y_valid, y_pred, average = 'weighted'))
        recall.append(recall_score(y_valid, y_pred, average = 'weighted'))
        f1.append(f1_score(y_valid, y_pred, average='weighted'))


    # get arrays
    accuracy_set = np.array(accuracy)
    precision_set = np.array(precision)
    recall_set = np.array(recall)
    f1_set = np.array(f1)

    print("Mean Accuracy: {}".format(accuracy_set.mean()))
    print("Mean Precision: {}".format(precision_set.mean()))
    print("Mean Recall: {}".format(recall_set.mean()))
    print("Mean F1-Score: {}".format(f1_set.mean()))
    return accuracy_set.mean()

KNN

In [7]:
from sklearn.neighbors import KNeighborsClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters

KNN_Accuracies = []
for i in range (1,20):
  knn = KNeighborsClassifier(n_neighbors=i) # vary n_neighbours from 1 to 20
  KNN_Accuracies.append(cross_validation(knn))


Mean Accuracy: 0.602555168408827
Mean Precision: 0.6131151504322236
Mean Recall: 0.602555168408827
Mean F1-Score: 0.5972500860118946
Mean Accuracy: 0.5644599303135889
Mean Precision: 0.6140614179797413
Mean Recall: 0.5644599303135889
Mean F1-Score: 0.561419035071033
Mean Accuracy: 0.5689895470383275
Mean Precision: 0.5697572501698472
Mean Recall: 0.5689895470383275
Mean F1-Score: 0.5529958278442841
Mean Accuracy: 0.5643437862950058
Mean Precision: 0.5685522551495411
Mean Recall: 0.5643437862950058
Mean F1-Score: 0.5580131299861643
Mean Accuracy: 0.5451800232288038
Mean Precision: 0.5525511875112203
Mean Recall: 0.5451800232288038
Mean F1-Score: 0.5398530951386461
Mean Accuracy: 0.5213704994192799
Mean Precision: 0.524556265996299
Mean Recall: 0.5213704994192799
Mean F1-Score: 0.5144078839193692
Mean Accuracy: 0.48792102206736354
Mean Precision: 0.4859579490067295
Mean Recall: 0.48792102206736354
Mean F1-Score: 0.482504873498371
Mean Accuracy: 0.47351916376306613
Mean Precision: 0.48141

In [8]:
# Print the accuracies
for i in KNN_Accuracies:
  print(i)

0.602555168408827
0.5644599303135889
0.5689895470383275
0.5643437862950058
0.5451800232288038
0.5213704994192799
0.48792102206736354
0.47351916376306613
0.47351916376306624
0.4638792102206736
0.44006968641114985
0.4159117305458769
0.43031358885017423
0.4015098722415796
0.39663182346109177
0.4300813008130081
0.41567944250871075
0.39698025551684085
0.40209059233449473


Random Forest

In [9]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters

n_estimators = np.arange(10, 101, 10)
max_depths = np.arange(1, 15, 1)

best_accuracy = []

for i in n_estimators:
  temp = []
  for j in max_depths:
    rf = RandomForestClassifier(n_estimators=i, max_depth = j)
    temp.append(cross_validation(rf))
  best_accuracy.append(temp)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Mean Accuracy: 0.4592334494773519
Mean Precision: 0.551601276665839
Mean Recall: 0.4592334494773519
Mean F1-Score: 0.3819306801778485
Mean Accuracy: 0.5409988385598141
Mean Precision: 0.6325032389379922
Mean Recall: 0.5409988385598141
Mean F1-Score: 0.504675668182003
Mean Accuracy: 0.6842044134727062
Mean Precision: 0.7095822351606325
Mean Recall: 0.6842044134727062
Mean F1-Score: 0.672411013956988
Mean Accuracy: 0.6603948896631825
Mean Precision: 0.6889712700830648
Mean Recall: 0.6603948896631825
Mean F1-Score: 0.6509555454024727
Mean Accuracy: 0.7225319396051104
Mean Precision: 0.7459111202386464
Mean Recall: 0.7225319396051104
Mean F1-Score: 0.7185369699067907
Mean Accuracy: 0.6989547038327526
Mean Precision: 0.7324867133679595
Mean Recall: 0.6989547038327526
Mean F1-Score: 0.6942847752107513
Mean Accuracy: 0.7274099883855982
Mean Precision: 0.7557998309635023
Mean Recall: 0.7274099883855982
Mean F1-Score: 0.7222376187976034
Mean Accuracy: 0.7606271777003485
Mean Precision: 0.778079

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Mean Accuracy: 0.48362369337979094
Mean Precision: 0.5228133748138555
Mean Recall: 0.48362369337979094
Mean F1-Score: 0.38847863770783836
Mean Accuracy: 0.5595818815331011
Mean Precision: 0.6474714371746892
Mean Recall: 0.5595818815331011
Mean F1-Score: 0.5394421665372383
Mean Accuracy: 0.645993031358885
Mean Precision: 0.6827417763382011
Mean Recall: 0.645993031358885
Mean F1-Score: 0.6413431125837566
Mean Accuracy: 0.665156794425087
Mean Precision: 0.6920993814687952
Mean Recall: 0.665156794425087
Mean F1-Score: 0.6617388886800052
Mean Accuracy: 0.7272938443670152
Mean Precision: 0.739487463638047
Mean Recall: 0.7272938443670152
Mean F1-Score: 0.7206099465302366
Mean Accuracy: 0.7418118466898955
Mean Precision: 0.7566630151035811
Mean Recall: 0.7418118466898955
Mean F1-Score: 0.7414408161144712
Mean Accuracy: 0.7463414634146341
Mean Precision: 0.7710560920628773
Mean Recall: 0.7463414634146341
Mean F1-Score: 0.7450983320209219
Mean Accuracy: 0.7844367015098722
Mean Precision: 0.80789

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

Mean Accuracy: 0.45946573751451797
Mean Precision: 0.5207295361563654
Mean Recall: 0.45946573751451797
Mean F1-Score: 0.3572149814709694
Mean Accuracy: 0.5932636469221835
Mean Precision: 0.6788853088223662
Mean Recall: 0.5932636469221835
Mean F1-Score: 0.5735168860366215
Mean Accuracy: 0.6795586527293844
Mean Precision: 0.7269653016360333
Mean Recall: 0.6795586527293844
Mean F1-Score: 0.6781166019404314
Mean Accuracy: 0.6984901277584206
Mean Precision: 0.7367451163440407
Mean Recall: 0.6984901277584206
Mean F1-Score: 0.6935187463855104
Mean Accuracy: 0.732288037166086
Mean Precision: 0.7583688536266934
Mean Recall: 0.732288037166086
Mean F1-Score: 0.7297414193245177
Mean Accuracy: 0.7797909407665505
Mean Precision: 0.8003073172149827
Mean Recall: 0.7797909407665505
Mean F1-Score: 0.778053496396216
Mean Accuracy: 0.7941927990708478
Mean Precision: 0.8163114499500029
Mean Recall: 0.7941927990708478
Mean F1-Score: 0.793541763265688
Mean Accuracy: 0.8133565621370499
Mean Precision: 0.83489

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Mean Accuracy: 0.4645760743321719
Mean Precision: 0.561479006862282
Mean Recall: 0.4645760743321719
Mean F1-Score: 0.3693885488325182
Mean Accuracy: 0.6076655052264809
Mean Precision: 0.6938786788808087
Mean Recall: 0.6076655052264809
Mean F1-Score: 0.5890885377069114
Mean Accuracy: 0.6699186991869919
Mean Precision: 0.7176554458519313
Mean Recall: 0.6699186991869919
Mean F1-Score: 0.6625059797595394
Mean Accuracy: 0.7225319396051104
Mean Precision: 0.7584893479790785
Mean Recall: 0.7225319396051104
Mean F1-Score: 0.7203207503890754
Mean Accuracy: 0.751219512195122
Mean Precision: 0.775133279296859
Mean Recall: 0.751219512195122
Mean F1-Score: 0.749342263103964
Mean Accuracy: 0.7845528455284553
Mean Precision: 0.8068688390448313
Mean Recall: 0.7845528455284553
Mean F1-Score: 0.7812914506688103
Mean Accuracy: 0.7989547038327526
Mean Precision: 0.8180785090590422
Mean Recall: 0.7989547038327526
Mean F1-Score: 0.7964583931918151
Mean Accuracy: 0.8131242740998837
Mean Precision: 0.82710421

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Mean Accuracy: 0.4550522648083623
Mean Precision: 0.4694282450169118
Mean Recall: 0.4550522648083623
Mean F1-Score: 0.3543989047167366
Mean Accuracy: 0.6030197444831592
Mean Precision: 0.685447867801962
Mean Recall: 0.6030197444831592
Mean F1-Score: 0.5856674694469941
Mean Accuracy: 0.68931475029036
Mean Precision: 0.7404166458082532
Mean Recall: 0.68931475029036
Mean F1-Score: 0.6831153985767219
Mean Accuracy: 0.7132404181184671
Mean Precision: 0.7475628342492454
Mean Recall: 0.7132404181184671
Mean F1-Score: 0.7113285194134269
Mean Accuracy: 0.7368176538908247
Mean Precision: 0.7625510237894858
Mean Recall: 0.7368176538908247
Mean F1-Score: 0.734548887617531
Mean Accuracy: 0.7799070847851336
Mean Precision: 0.8026100105932038
Mean Recall: 0.7799070847851336
Mean F1-Score: 0.7758747247454345
Mean Accuracy: 0.7894308943089431
Mean Precision: 0.808045443773876
Mean Recall: 0.7894308943089431
Mean F1-Score: 0.7874886645736703
Mean Accuracy: 0.8229965156794424
Mean Precision: 0.8419744960

In [10]:
# Print the accuracies
for row in best_accuracy:
  print("\t".join(map(str, row))) # Convert each element to string and join with tabs

0.4592334494773519	0.5409988385598141	0.6842044134727062	0.6603948896631825	0.7225319396051104	0.6989547038327526	0.7274099883855982	0.7606271777003485	0.7656213704994193	0.7506387921022066	0.7267131242740998	0.7655052264808362	0.8040650406504065	0.7844367015098722
0.48362369337979094	0.5595818815331011	0.645993031358885	0.665156794425087	0.7272938443670152	0.7418118466898955	0.7463414634146341	0.7844367015098722	0.8038327526132404	0.8227642276422763	0.770383275261324	0.7990708478513356	0.7989547038327526	0.8228803716608594
0.483855981416957	0.5742160278745645	0.645876887340302	0.7032520325203253	0.7462253193960511	0.770267131242741	0.7653890824622531	0.8036004645760743	0.8180023228803716	0.7749128919860627	0.7890824622531939	0.7846689895470383	0.7943089430894309	0.7845528455284553
0.4883855981416957	0.5645760743321719	0.6077816492450638	0.6938443670150987	0.7653890824622532	0.7845528455284553	0.7797909407665505	0.770383275261324	0.7799070847851336	0.8088269454123113	0.789198606271777	

MLP

In [11]:
import numpy as np
from sklearn.neural_network import MLPClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters

learning_rate = [0.01, 0.005, 0.001, 0.0005, 0.0001]
epoch = np.arange(10, 210, 20)

best_accuracy = []

for i in epoch:
  temp = []
  for j in learning_rate:
    mlp = MLPClassifier(hidden_layer_sizes=(100,500), learning_rate_init=0.001, max_iter=20) # vary learning rate as 0.0001, 0.001, 0.01, 0.05, 0.1, 1 and max_iter from 10 to 100
    temp.append(cross_validation(mlp))
  best_accuracy.append(temp)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contro

Mean Accuracy: 0.33495934959349594
Mean Precision: 0.16899194750672092
Mean Recall: 0.33495934959349594
Mean F1-Score: 0.21117681190966583


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Ma

Mean Accuracy: 0.3497096399535424
Mean Precision: 0.24099303810360154
Mean Recall: 0.3497096399535424
Mean F1-Score: 0.20733505137275787


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contro

...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and bein

Mean Accuracy: 0.40162601626016253
Mean Precision: 0.16413577923733444
Mean Recall: 0.40162601626016253
Mean F1-Score: 0.23225352283329093


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and bein

Mean Accuracy: 0.3586527293844367
Mean Precision: 0.215671247755302
Mean Recall: 0.3586527293844367
Mean F1-Score: 0.24653518958722556


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
# Print the accuracies
for row in best_accuracy:
  print("\t".join(map(str, row))) # Convert each element to string and join with tabs

0.33495934959349594	0.3497096399535424	0.32590011614401854	0.3878048780487805	0.378281068524971
0.3153310104529617	0.3010452961672474	0.30685249709639956	0.3444831591173055	0.4063879210220674
0.3641114982578397	0.39732868757259	0.31114982578397216	0.3545876887340302	0.3058072009291522
0.3878048780487805	0.4260162601626016	0.2962833914053426	0.36399535423925666	0.3348432055749129
0.3401858304297328	0.3439024390243902	0.37723577235772354	0.3687572590011614	0.31080139372822296
0.2921022067363531	0.334959349593496	0.34494773519163763	0.3354239256678281	0.3873403019744483
0.3878048780487805	0.39698025551684085	0.3590011614401858	0.2825783972125436	0.3439024390243902
0.37781649245063875	0.29686411149825787	0.2914053426248548	0.373054587688734	0.3735191637630662
0.3399535423925668	0.29732868757259	0.3735191637630662	0.34471544715447155	0.3878048780487805
0.3401858304297328	0.3592334494773519	0.3778164924506388	0.40162601626016253	0.3586527293844367


Retrain the best model

In [13]:
best_model = RandomForestClassifier(n_estimators=50, max_depth = 10)
best_model.fit(X, Y)

RandomForestClassifier(max_depth=10, n_estimators=50)

Save the best model

In [14]:
import pickle
fh = open("/kaggle/working/rf_best_model.pkl", "wb")
pickle.dump(best_model, fh)
fh.close()